# Task 1: Data Enrichment
## Forecasting Financial Inclusion in Ethiopia

**Objective**
- Enrich the unified dataset with additional observations, events, and impact links
- Improve explanatory power for Access and Usage forecasting
- Maintain strict compliance with the unified schema

**Design Principle**
Events are NOT assigned to pillars.  
Their effects are modeled via `impact_link` records.

# Import Libraries

In [1]:
import pandas as pd
from datetime import datetime

# Load Raw Dataset

In [2]:
DATA_PATH = "../data/raw/"
OUTPUT_PATH = "../data/processed/"

fi_df = pd.read_csv(DATA_PATH + "ethiopia_fi_unified_data.csv")

fi_df.shape

(43, 1)

# Separate Record Types

In [3]:
# If the CSV was read with the wrong separator it may be a single-column DF where the column name
# contains all headers joined by tabs. Fix that and then split into record-type dataframes.
if "record_type" not in fi_df.columns:
    combined_header = fi_df.columns[0]
    cols = [c.strip() for c in combined_header.split("\t")]
    fi_df = fi_df.iloc[:, 0].astype(str).str.split("\t", expand=True)
    fi_df.columns = cols

obs_df = fi_df[fi_df["record_type"] == "observation"].copy()
events_df = fi_df[fi_df["record_type"] == "event"].copy()
impact_df = fi_df[fi_df["record_type"] == "impact_link"].copy()
targets_df = fi_df[fi_df["record_type"] == "target"].copy()
events_df = fi_df[fi_df["record_type"] == "event"].copy()
impact_df = fi_df[fi_df["record_type"] == "impact_link"].copy()
targets_df = fi_df[fi_df["record_type"] == "target"].copy()

# Define Metadata Defaults

In [4]:
COLLECTED_BY = "Mulatie Kindie"
COLLECTION_DATE = datetime.today().strftime("%Y-%m-%d")
CONFIDENCE_DEFAULT = "medium"

# ADD NEW OBSERVATIONS

### Rationale for New Observations

Given sparse Findex data, forecasting requires **proxy indicators** that:
- Move annually or quarterly
- Are causally linked to Access or Usage
- Reflect Ethiopia-specific constraints

We prioritize:
- Smartphone penetration
- Mobile broadband coverage
- Agent density
- Active mobile money usage


## Create New Observation Records

In [5]:
new_observations = [
    {
        "record_type": "observation",
        "pillar": "usage",
        "indicator": "Smartphone penetration rate",
        "indicator_code": "ENAB_SMARTPHONE_PCT",
        "value_numeric": 44.0,
        "observation_date": "2023-12-31",
        "source_name": "GSMA Mobile Economy Sub-Saharan Africa",
        "source_url": "https://www.gsma.com/mobileeconomy/",
        "confidence": "medium",
        "original_text": "Smartphone adoption in Ethiopia reached approximately 44% in 2023.",
        "collected_by": COLLECTED_BY,
        "collection_date": COLLECTION_DATE,
        "notes": "Smartphone access is a binding constraint for advanced digital payment usage."
    },
    {
        "record_type": "observation",
        "pillar": "usage",
        "indicator": "4G population coverage",
        "indicator_code": "ENAB_4G_COVERAGE_PCT",
        "value_numeric": 35.0,
        "observation_date": "2023-12-31",
        "source_name": "GSMA / ITU",
        "source_url": "https://www.itu.int/",
        "confidence": "medium",
        "original_text": "4G networks cover approximately 35% of Ethiopia’s population.",
        "collected_by": COLLECTED_BY,
        "collection_date": COLLECTION_DATE,
        "notes": "Digital payments beyond P2P require reliable mobile broadband."
    }
]


## Append Observations Safely

In [6]:
new_obs_df = pd.DataFrame(new_observations)

# Use a copy of obs_df with unique column names for safe concatenation (handles duplicate empty columns)
_obs_for_concat = obs_df.copy()
_obs_for_concat.columns = [
	c if c != "" else f"_empty_col_{i}" for i, c in enumerate(_obs_for_concat.columns)
]

# Reindex the new observations to match the concat-ready columns
new_obs_df_reindexed = new_obs_df.reindex(columns=_obs_for_concat.columns)

obs_df_enriched = pd.concat([_obs_for_concat, new_obs_df_reindexed], ignore_index=True)

obs_df_enriched.shape

(32, 36)

# ADD NEW EVENTS

## Event Selection Logic

### Why Add These Events?

Forecasting requires modeling **structural breaks**, such as:
- Interoperability
- Digital ID rollout
- Regulatory liberalization

These do not directly increase inclusion, but **enable mechanisms** that do.

## Create New Event Records

In [7]:
new_events = [
    {
        "record_type": "event",
        "id": "EVT_INTEROP_2022",
        "event_name": "National mobile money interoperability rollout",
        "category": "infrastructure",
        "event_date": "2022-12-01",
        "source_name": "National Bank of Ethiopia",
        "source_url": "https://www.nbe.gov.et/",
        "confidence": "high",
        "original_text": "Ethiopia launched interoperable digital payment infrastructure in 2022.",
        "collected_by": COLLECTED_BY,
        "collection_date": COLLECTION_DATE,
        "notes": "Interoperability reduces friction in P2P and merchant payments."
    }
]


## Append Events

In [8]:
new_events_df = pd.DataFrame(new_events)

# Ensure unique column names for safe concatenation
_events_for_concat = events_df.copy()
_events_for_concat.columns = [
	c if c != "" else f"_empty_col_{i}" for i, c in enumerate(_events_for_concat.columns)
]

# Reindex new_events_df to match columns of _events_for_concat
new_events_df_reindexed = new_events_df.reindex(columns=_events_for_concat.columns)

events_df_enriched = pd.concat([_events_for_concat, new_events_df_reindexed], ignore_index=True)

events_df_enriched.shape

(11, 36)

# ADD IMPACT LINKS

## Impact Modeling Logic

### Impact Link Design

Impact links translate events into **quantified effects** on indicators.

Each link specifies:
- Which indicator is affected
- Direction of effect
- Relative magnitude
- Time lag
- Evidence basis


## Create Impact Links

In [9]:
new_impact_links = [
    {
        "record_type": "impact_link",
        "parent_id": "EVT_INTEROP_2022",
        "pillar": "usage",
        "related_indicator": "USG_DIGITAL_PAYMENT",
        "impact_direction": "positive",
        "impact_magnitude": 0.10,
        "lag_months": 6,
        "evidence_basis": "CGAP studies show interoperability increases digital payment usage by reducing network effects."
    }
]

## Append Impact Links

In [10]:
new_impact_df = pd.DataFrame(new_impact_links)

# Ensure unique column names in the existing impact_df (handle empty duplicate column names)
_impact_for_concat = impact_df.copy()
_impact_for_concat.columns = [
	c if c != "" else f"_empty_col_{i}" for i, c in enumerate(_impact_for_concat.columns)
]

# Reindex new impact rows to match concat-ready columns
new_impact_df_reindexed = new_impact_df.reindex(columns=_impact_for_concat.columns)

impact_df_enriched = pd.concat([_impact_for_concat, new_impact_df_reindexed], ignore_index=True)

impact_df_enriched.shape

C:\Users\mulat\AppData\Local\Temp\ipykernel_15836\1019972307.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  impact_df_enriched = pd.concat([_impact_for_concat, new_impact_df_reindexed], ignore_index=True)


(1, 36)

# REASSEMBLE UNIFIED DATASET

## Combine All Records

In [12]:
def _ensure_unique_columns(df):
    cols = list(df.columns)
    new_cols = []
    seen = {}
    for i, c in enumerate(cols):
        name = f"_empty_col_{i}" if pd.isna(c) or c == "" else str(c)
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        new_cols.append(name)
    out = df.copy()
    out.columns = new_cols
    return out

fi_enriched = pd.concat([
    _ensure_unique_columns(obs_df_enriched),
    _ensure_unique_columns(events_df_enriched),
    _ensure_unique_columns(impact_df_enriched),
    _ensure_unique_columns(targets_df)
], ignore_index=True)

fi_enriched.shape

(47, 36)

## Save Enriched Dataset

In [13]:
OUTPUT_FILE = OUTPUT_PATH + "ethiopia_fi_unified_data_enriched.csv"

fi_enriched.to_csv(OUTPUT_FILE, index=False)

OUTPUT_FILE

'../data/processed/ethiopia_fi_unified_data_enriched.csv'

# DATA ENRICHMENT LOG